# MLP with `torch.nn`

While there are plenty of tutorials on the usage of `torch.nn`, this is another summary which is largely equivalent to the low level implementation, but much more compact.

In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

from idl.common import ClassifierTrainer, count_parameters, get_datasets_and_loaders
from idl.week1.analysis import plot_learning_curves

## Data

The `torchvision` package provides some popular datasets for quick access.

This version does not come with a validation set. We now have 60,000 training samples and 10,000 test samples. Best practice would be to take about 10,000 samples out of the training set and create a separate validation set.
We are not doing this here. Technically this is bad practice! Over time, this could lead to "overfitting to the test set" if we make decisions about model architecture, hyperparameters etc. based on the test set performance.

**NOTE** the code below showcases how to load data using `torchvision` with "minimal effort".
We also provide a `get_datasets_and_loaders` function that is more fully featured.

In [ ]:
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor())

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor())

Data loaders supply convenient access to batches of data. Note the `num_workers` argument. This is very important! It sets the number of CPU processes dedicated to preparing data. The default is 0, which results in data being prepared by the main process. This can severly bottleneck our training! The GPU will spend most of the time idling, waiting for data to be supplied. 

Experiment with this and see what works best on your hardware. On our server, we found
- 1 worker makes no difference -- 0 or 1 take about 5 seconds per epoch with the given model and batch size
- 2 workers already cut time in half!
- More workers improve time further, but there are diminishing returns. E.g. 8 workers lead to around 1.2-1.5 seconds per epoch.

But: 8 processes may be fine for a powerful compute server, but could be too much for e.g. a mid-range laptop CPU. Using too many workers may impact your overall system performance!

For comparison: The low-level implementation where we put the whole dataset onto the GPU only takes around 0.5 seconds per epoch. In principle we could implement our own `Dataset` where we move the full dataset to the GPU.

In [ ]:
batch_size = 128

# Create data loaders.
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True,
                              num_workers=8)
test_dataloader = DataLoader(test_data, batch_size=batch_size,
                             num_workers=8)

input_batch, label_batch = next(iter(test_dataloader))
print(f"Shape of X [N, C, H, W]: {input_batch.shape}")
print(f"Shape of y: {label_batch.shape} {label_batch.dtype}")

With our pre-made function, the notebook only needs to call it -- the two code cells above become superfluous:

In [ ]:
batch_size = 128
train_data, test_data, train_dataloader, test_dataloader = get_datasets_and_loaders("mnist", batch_size=batch_size, num_workers=8)

## Model

Creating models is simple with `torch.nn`. Many models are simply layers applied in sequence, and this is best done with a `Sequential` module. Note that modules often do not map onto the "layers" we talk about in class. For example, activation functions are usually separate modules.

Modules with trainable weights, such as `Linear`, will usually have some standard schema to initialize them. The cell below shows an example of how to overwrite this using `apply`. Printing the maximum weight before and after this operation clearyl shows that the weights have been changed.

An important note here -- no softmax in the model! This is because the cross-entropy loss, defined further below, has the log-softmax built-in to the loss function itself. So we do not apply softmax ourselves -- this would be incorrect!

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

hidden_dim = 512

model = nn.Sequential(nn.Flatten(),
                      nn.Linear(784, hidden_dim),
                      nn.GELU(),
                      nn.Linear(hidden_dim, 10)).to(device)
print(model)
with torch.no_grad():
    print("Maximum weight before custom init: ", model[1].weight.max())


def glorot_init(layer: nn.Module):
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)


with torch.no_grad():
    model.apply(glorot_init)
    print("Maximum weight after custom init", model[1].weight.max())

In [ ]:
n_params = count_parameters(model)
print(f"Model with {n_params:,d} parameters.")

## Training

We have refactored our previous training code into a `Trainer` class.
**Be sure to look at the implementation in detail!**
It's actually very similar to our previous low-level training loop, except for using loss functions and optimizers from the `nn` package.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
trainer = ClassifierTrainer(model=model, optimizer=optimizer,
                            training_loader=train_dataloader, validation_loader=test_dataloader,
                            n_epochs=40, device=device)

In [ ]:
metrics = trainer.train_model()

In [ ]:
plot_learning_curves(metrics, keys=["cross_entropy", "accuracy"])

In [ ]:
trainer.plot_examples()

In [ ]:
def visualize_features(colormap: str = "local"):
    if colormap not in ["local", "global"]:
        raise ValueError("colormap argument should be 'local' or 'global'")
    features = model[1].weight.detach().cpu().numpy()
    global_absmax = abs(features).max()
    
    plt.figure(figsize=(12, 6))
    for ind, pattern in enumerate(features):
        plt.subplot(16, 32, ind+1)
        absmax = abs(pattern).max() if colormap == "local" else global_absmax
        plt.imshow(pattern.reshape(28, 28), vmin=-absmax, vmax=absmax, cmap="RdBu_r")
        plt.axis("off")
        #plt.colorbar()
    plt.suptitle("First layer features with {} colormaps".format(colormap))
    plt.show()

In [ ]:
visualize_features("local")
visualize_features("global")

## Model Calibration

Our classifier outputs a probability for each class. How useful are these values really? Ideally, the probability should correspond samewhat to the chance that this prediction is correct. If this is the case, the model is called _calibrated_. For example, if the output is something like 0.999, the model best be correct! If the output is, say, 0.9, the model should be correct in 90% of the cases.

We can test this by, for example, collecting all outputs with a probability of around 0.9, and checking how often the model is correct. As the outputs are rarely nice round numbers, we bin the outputs, in the example below in steps of 0.05. Then we plot, for each bin, the proportion of correct responses for that bin.

In [ ]:
all_preds = []
all_corrects = []
with torch.no_grad():
    for input_batch, label_batch in test_dataloader:
        input_batch = input_batch.to(device)
        label_batch = label_batch.to(device)
        predictions = torch.nn.functional.softmax(model(input_batch), dim=1)
        guess = predictions.argmax(axis=1)
        guess_probs = torch.gather(predictions, 1, guess[:, None])[:, 0]
        correct = (guess == label_batch).type(torch.float)

        all_preds.append(guess_probs.cpu().numpy())
        all_corrects.append(correct.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_corrects = np.concatenate(all_corrects)

In [ ]:
plt.hist(all_preds, bins=100)
plt.title("Histogram of probability output for predicted class")
plt.xlabel("Probability")
plt.ylabel("Count")
plt.show()

In [ ]:
step = 0.05
collection = {ind: [] for ind in range(0, int(1/step))}
for pred, correct in zip(all_preds, all_corrects):
    if pred == 1:
        pred = 0.9999 # prediction of 1 would result in its own bin otherwise...
    calibration_bin = int(pred / step)
    collection[calibration_bin].append(correct)

In [ ]:
midpoints = []
actual = []
for ind, corrects in collection.items():
    lower = ind * step
    upper = (ind + 1) * step
    middle = lower + step / 2
    
    midpoints.append(middle)
    actual.append(np.mean(corrects))

The common wisdom is that neural networks tend to be over-confident:
The rate of correctness is often lower than the probability output.
For example, when the model outputs 0.8 for some class, it's correct in less than 80% of the cases.
Visually, the blue observed line is under/to the right of the ideal orange line.

This will depend on many factors; for example, a more overfitted model may be more overconfident.
Conversely, training for fewer epochs can result in an "underconfident" model.
These results are also quite noisy as the model is almost always very confident, so there are few data points for the lower probability outputs.

In [ ]:
plt.plot(midpoints, actual, label="Observed")
plt.plot(midpoints, midpoints, "--", label="Ideal calibration")

plt.xlabel("Probability output")
plt.ylabel("Proportion of correct responses")
plt.title("Model calibration")
plt.legend()
plt.show()